In [1]:
from IPython.display import clear_output
from Fetch_Live_Data import *
from Trade_Selection_All import *
from Chandelier_ZLSMA import *
from Trade_Execution import *

In [2]:
def pick_best_coin():
    scalping_filter = BinanceAllUSDTScalpingFilter(
        max_workers=8,
        delay_between_requests=0.1,
        weight_profile='volatile'  # Options: 'balanced', 'volatile', 'trending'
    )

    print("Scanning ALL Binance USDT pairs for scalping opportunities...")

    # get a list of dicts (or records)
    best_coins = scalping_filter.filter_all_usdt_pairs(
        min_volume=50_000,
        top_n=30,
        use_parallel=True,
        volume_filter_first=False,
        use_percentile_volume=True
    )

    # turn into a DataFrame
    df = pd.DataFrame(best_coins)

    # base filters: uptrend + cheap coins
    base_mask = (df['trend_direction'] == 1) & (df['current_price'] <= 50)

    # try successive RSI thresholds
    for rsi_limit in (40, 50, 60, 65):
        candidates = df[base_mask & (df['rsi'] <= rsi_limit)]
        if len(candidates) >= 3:
            # return the top 3 symbols with highest buy_pressure as a list
            return candidates.nlargest(3, 'buy_pressure')['symbol'].tolist()
        elif not candidates.empty:
            # if we have some candidates but less than 3, return all available
            return candidates.nlargest(len(candidates), 'buy_pressure')['symbol'].tolist()

    # nothing matched
    return []

In [ ]:
# Define your function to fetch, calculate, and merge the data
def fetch_and_process_data(symbol):
    df = get_single_fetch(symbol, 500)  # Fetch data
    df_zlsma = calculate_zlsma(df, close_column='close', length=200)  # Calculate ZLSMA
    df_chandelier = calculate_chandelier(df, atr_period=1, atr_multiplier=2.0)  # Calculate Chandelier
    merged_chandelier_zlsma = merge_zlsma_chandelier(df_zlsma, df_chandelier)  # Merge ZLSMA and Chandelier
    merged_chandelier_zlsma = merged_chandelier_zlsma[["timestamp", "close", "zlsma_200", "buy_signal", "sell_signal"]]  # Filter relevant columns
    return merged_chandelier_zlsma[480:]

def run_task():
    # Get the latest symbols every 2 hours
    symbols = pick_best_coin()
    return symbols

def fetch_and_display_data_(symbols):
    # Run the fetch_and_process_data every 10 seconds to display results
    result = fetch_and_process_data(symbols)
    clear_output(wait=True)  # Clear previous output in Jupyter Notebook
    print("Monitoring on: ", symbols)
    print(result)  # Display the new result
    print("\n")
    
    return result

# Main loop that runs every 2 hours
while True:
    symbols = run_task()  # Get the symbols every 2 hours (list of 3 symbols)
    print("New symbols received. Monitoring begins...\n")
    print(f"Symbols to monitor: {symbols}")
    
    trade_taken = False  # Flag to track if any trade was taken
    
    # Try each symbol one by one
    for i, symbol in enumerate(symbols):
        print(f"\n--- Checking symbol {i+1}/3: {symbol} ---")
        
        # Continuous 10-second updates with fetched data for current symbol
        symbol_checked = False
        
        while not symbol_checked:
            out = fetch_and_display_data_([symbol])  # Pass single symbol as list
            
            # Check if the DataFrame is not empty and get the last row
            if not out.empty:
                # Access the last row using iloc[-1]
                if out['close'].iloc[-1] > out['zlsma_200'].iloc[-1] and out['buy_signal'].iloc[-1] == True:
                    print(f"Buy signal detected for symbol: {symbol}")
                    
                    if out['buy_signal'].sum() <= 15:
                        #bot = SimpleATRTradingBot()
                        #result = bot.buy_signal(symbol, 10)
                        #status = bot.get_position_status()
                        print(f"✅ Trade Taken: {symbol}")
                        print("Running trade for 2 hours...")
                        trade_taken = True
                        symbol_checked = True  # Move to next phase
                        break  # Exit the symbol checking loop
                    else:
                        print(f"Buy signal sum > 15 for {symbol}, trying next symbol...")
                        symbol_checked = True  # Try next symbol
                else:
                    print(f"No buy signal detected for symbol: {symbol}")
                    symbol_checked = True  # Try next symbol
            else:
                print(f"The DataFrame is empty for {symbol}. No data available.")
                symbol_checked = True  # Try next symbol
        
        # If trade was taken, break out of symbol iteration
        if trade_taken:
            break
    
    # If no trade was taken with any symbol, refresh for new symbols
    if not trade_taken:
        print("\n❌ No trades taken with any symbols. Refreshing for new symbols...")
        continue  # Skip the 2-hour sleep and get new symbols immediately
    
    # If trade was taken, monitor for 2 hours with 10-second intervals
    print(f"\n🔄 Monitoring trade for 2 hours...")
    monitoring_start = time.time()
    
    while time.time() - monitoring_start < 2 * 60 * 60:  # 2 hours
        # You can add monitoring logic here if needed
        time.sleep(10)  # Wait for 10 seconds before next check
    
    print("2-hour monitoring period completed. Getting new symbols...")

Monitoring on:  ['ETCUSDT']
              timestamp  close  zlsma_200  buy_signal  sell_signal
480 2025-06-13 16:30:00  16.34  16.056220           1            0
481 2025-06-13 16:45:00  16.34  16.041298           1            0
482 2025-06-13 17:00:00  16.48  16.032903           1            0
483 2025-06-13 17:15:00  16.42  16.022089           1            0
484 2025-06-13 17:30:00  16.37  16.009681           1            0
485 2025-06-13 17:45:00  16.38  16.002187           1            0
486 2025-06-13 18:00:00  16.36  15.991142           1            0
487 2025-06-13 18:15:00  16.34  15.979905           1            0
488 2025-06-13 18:30:00  16.31  15.968681           1            0
489 2025-06-13 18:45:00  16.35  15.958276           1            0
490 2025-06-13 19:00:00  16.29  15.947477           1            0
491 2025-06-13 19:15:00  16.32  15.936905           1            0
492 2025-06-13 19:30:00  16.37  15.928697           1            0
493 2025-06-13 19:45:00  16.39  15

In [ ]:
bot = SimpleATRTradingBot()
result = bot.buy_signal('BTC', 1000)
status = bot.get_position_status()